# 01 — Data Structure and Quality Audit

**Person 1 deliverable** — Wisconsin Diagnostic Breast Cancer (WDBC) dataset

**Main responsibility:** Confirm the dataset was imported correctly and contains valid, usable information.

**Source files:**
- `wdbc.data` — raw observations (no header row)
- `wdbc.names` — dataset documentation and attribute definitions

**Expected dataset profile:**

| Property | Expected value |
|---|---|
| Observations | 569 |
| Columns | 32 (1 ID + 1 diagnosis + 30 features) |
| Missing values | 0 |
| Duplicate rows | 0 |
| Duplicate IDs | 0 |
| Diagnosis labels | `M`, `B` only |
| Benign (B) | 357 (62.7%) |
| Malignant (M) | 212 (37.3%) |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(".")
RAW_FILE = DATA_DIR / "wdbc.data"
NAMES_FILE = DATA_DIR / "wdbc.names"
OUTPUT_FILE = DATA_DIR / "wdbc_clean.csv"

ID_COL = "id"
TARGET_COL = "diagnosis"

## 1. Assign column names from `wdbc.names`

Per `wdbc.names` (attributes 1–32):
1. **ID number** — anonymous patient identifier
2. **Diagnosis** — `M` = malignant, `B` = benign
3–32. **30 real-valued features** — mean, standard error, and worst values for 10 nucleus measurements (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension)

> **Important:** `wdbc.data` has **no header row**. We must pass `header=None` and supply names manually, or pandas will treat the first patient as column names.

In [2]:
# Column names derived from wdbc.names attribute information (fields 1–32)
COLUMNS = [
    "id", "diagnosis",
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "smoothness_mean", "compactness_mean", "concavity_mean",
    "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se",
    "smoothness_se", "compactness_se", "concavity_se",
    "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst",
    "smoothness_worst", "compactness_worst", "concavity_worst",
    "concave_points_worst", "symmetry_worst", "fractal_dimension_worst",
]

df = pd.read_csv(RAW_FILE, header=None, names=COLUMNS)
feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]

print(f"Loaded {RAW_FILE.name}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)} total → 1 ID, 1 diagnosis, {len(feature_cols)} features")

Loaded wdbc.data
Shape: (569, 32)
Columns: 32 total → 1 ID, 1 diagnosis, 30 features


## 2. Confirm expected shape and column breakdown

In [3]:
assert df.shape == (569, 32), f"Expected (569, 32), got {df.shape}"
assert len(feature_cols) == 30, f"Expected 30 features, got {len(feature_cols)}"

print("✓ Shape is 569 rows × 32 columns")
print("✓ Column breakdown: 1 ID + 1 diagnosis + 30 numerical features")
print("\nFirst 3 rows:")
df.head(3)

✓ Shape is 569 rows × 32 columns
✓ Column breakdown: 1 ID + 1 diagnosis + 30 numerical features

First 3 rows:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


## 3. Data quality audit

In [4]:
def audit_dataset(df, id_col=ID_COL, target_col=TARGET_COL):
    """Run all Person 1 quality checks and return a summary dict."""
    feature_cols = [c for c in df.columns if c not in [id_col, target_col]]
    numeric_df = df.select_dtypes(include="number")

    # Blank cells: empty strings or whitespace-only (not captured by isna)
    blank_mask = df.astype(str).apply(lambda s: s.str.strip().isin(["", "nan", "NaN", "NA"]))
    blank_counts = blank_mask.sum()
    blank_total = int(blank_counts.sum())

    # Constant columns (including empty)
    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]

    # Unexpected diagnosis labels
    valid_labels = {"M", "B"}
    unexpected_labels = sorted(set(df[target_col].unique()) - valid_labels)

    # Data types
    non_numeric_features = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]

    counts = df[target_col].value_counts()
    pct = (df[target_col].value_counts(normalize=True) * 100).round(1)

    report = {
        "shape": df.shape,
        "n_features": len(feature_cols),
        "missing_values": int(df.isna().sum().sum()),
        "blank_cells": blank_total,
        "blank_by_column": blank_counts[blank_counts > 0].to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_ids": int(df[id_col].duplicated().sum()),
        "infinite_values": int(np.isinf(numeric_df).sum().sum()),
        "constant_columns": constant_cols,
        "unexpected_diagnosis_labels": unexpected_labels,
        "non_numeric_features": non_numeric_features,
        "diagnosis_counts": counts.to_dict(),
        "diagnosis_pct": pct.to_dict(),
        "dtypes": df.dtypes.astype(str).to_dict(),
    }
    return report


report = audit_dataset(df)

print("=" * 55)
print("DATA QUALITY AUDIT")
print("=" * 55)
print(f"Shape:                    {report['shape']}")
print(f"Missing values (NaN):     {report['missing_values']}")
print(f"Blank cells:              {report['blank_cells']}")
print(f"Duplicate rows:           {report['duplicate_rows']}")
print(f"Duplicate IDs:            {report['duplicate_ids']}")
print(f"Infinite values:          {report['infinite_values']}")
print(f"Constant columns:         {report['constant_columns'] or 'none'}")
print(f"Unexpected diagnosis:     {report['unexpected_diagnosis_labels'] or 'none'}")
print(f"Non-numeric features:     {report['non_numeric_features'] or 'none'}")
print(f"Diagnosis counts:         {report['diagnosis_counts']}")
print(f"Diagnosis percentages:    {report['diagnosis_pct']}")
print("=" * 55)

DATA QUALITY AUDIT
Shape:                    (569, 32)
Missing values (NaN):     0
Blank cells:              0
Duplicate rows:           0
Duplicate IDs:            0
Infinite values:          0
Constant columns:         none
Unexpected diagnosis:     none
Non-numeric features:     none
Diagnosis counts:         {'B': 357, 'M': 212}
Diagnosis percentages:    {'B': 62.7, 'M': 37.3}


In [5]:
# Assert all quality checks pass
assert report["missing_values"] == 0, "Found missing values"
assert report["blank_cells"] == 0, "Found blank cells"
assert report["duplicate_rows"] == 0, "Found duplicate rows"
assert report["duplicate_ids"] == 0, "Found duplicate IDs"
assert report["infinite_values"] == 0, "Found infinite values"
assert report["constant_columns"] == [], f"Constant columns: {report['constant_columns']}"
assert report["unexpected_diagnosis_labels"] == [], f"Bad labels: {report['unexpected_diagnosis_labels']}"
assert report["non_numeric_features"] == [], f"Non-numeric features: {report['non_numeric_features']}"

assert report["diagnosis_counts"]["B"] == 357
assert report["diagnosis_counts"]["M"] == 212
assert abs(report["diagnosis_pct"]["B"] - 62.7) < 0.1
assert abs(report["diagnosis_pct"]["M"] - 37.3) < 0.1

print("✓ All quality checks passed")
print("✓ Diagnosis distribution: 357 B (62.7%), 212 M (37.3%)")

✓ All quality checks passed
✓ Diagnosis distribution: 357 B (62.7%), 212 M (37.3%)


In [6]:
# Inspect data types
pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values})

,column,dtype
0,id,int64
1,diagnosis,str
2,radius_mean,float64
3,texture_mean,float64
4,perimeter_mean,float64
5,area_mean,float64
6,smoothness_mean,float64
7,compactness_mean,float64
8,concavity_mean,float64
9,concave_points_mean,float64


## 4. Important finding

The raw WDBC dataset is already clean:
- **0 missing values** — no imputation needed
- **0 duplicate rows** — no rows to delete
- **0 duplicate IDs** — each patient appears once
- **Only valid diagnosis labels** (`M`, `B`)

No rows or columns need to be removed. Person 1's job is to **correctly label and validate** the import, then save a clean copy for downstream teammates.

## 5. Save clean dataset (`df_clean`)

In [7]:
# Enforce expected data types before saving
df_clean = df.copy()
df_clean[ID_COL] = df_clean[ID_COL].astype(int)
df_clean[TARGET_COL] = df_clean[TARGET_COL].astype(str)
for col in feature_cols:
    df_clean[col] = df_clean[col].astype(float)

# Re-run audit on the cleaned copy
final_report = audit_dataset(df_clean)
assert final_report["shape"] == (569, 32)

df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Saved df_clean → {OUTPUT_FILE}")
print(f"df_clean shape: {df_clean.shape}")
df_clean.head(3)

Saved df_clean → wdbc_clean.csv
df_clean shape: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
